# 第 7 章 · Dynamic agent workflows：用 Python 编排动态工作流

> 第 4 章我们用 `SequentialAgent / ParallelAgent / LoopAgent` 组装多智能体系统；第 5、6 章补齐了状态与观测。本章接着第 4 章的「版本前沿」，把 ADK 2.x 的 **Dynamic workflows** 做成可以逐格运行的教程。
>
> 本章目标：
> 1. 分清 **图编排、代码编排、LLM 委派**各自控制什么；
> 2. 用 `Workflow + @node + Context.run_node()` 写顺序、分支和迭代流程；
> 3. 在同一工作流中组合 **DeepSeek Agent 与普通 Python 函数**；
> 4. 掌握并发与限流、嵌套、重试与超时、人工确认与恢复、稳定 execution ID；
> 5. 理解 checkpoint 的边界，避免重复副作用与共享状态冲突。

**阅读建议**：先顺序读第 1～6 节建立模型，再读第 7～11 节高级机制，最后运行第 12 节综合案例。每节先解释「为什么」，再看代码和观察点。

**版本与运行约定**：官方标注该特性从 ADK Python **2.0.0** 开始提供；本章按仓库锁定的 **`google-adk==2.6.3` / Python 3.12** 验证。在线文档会继续演进，复制代码时优先核对本地版本。所有 LLM 示例仍使用 `LiteLlm(model="deepseek/deepseek-chat")`；其他示例无需 API Key。Notebook 原生支持顶层 `await`，不要再包 `asyncio.run()`。

官方入口：[Dynamic workflows](https://adk.dev/graphs/dynamic/) · [Graph workflows](https://adk.dev/graphs/) · [数据传递](https://adk.dev/graphs/data-handling/)。

---

## 1. 「动态」究竟在哪里？

把工作流想成一份厨房工单：**图编排**先把站点与路线画出来；**动态工作流**让厨师长执行一段 Python，看到检查结果后再决定重做、换步骤、同时开几口锅。两者都可以执行确定的规则，也都可以在节点中调用 LLM。

| 方式 | 执行路径如何表达 | 适用场景 | 与第 4 章的关系 |
|---|---|---|---|
| Template workflow agents | `Sequential / Parallel / Loop` 预设组合 | 常见流水线 | 前章使用的积木 |
| Graph-based workflows | 声明 nodes、edges 和路由 | 结构清晰、便于画图的流程 | 2.x 的 `Workflow` 图式 API |
| **Dynamic workflows** | **用 Python 条件、循环、函数调用安排子节点** | 分支多、迭代深、任务数取决于输入 | 本章的新编排方式 |
| LLM-driven delegation | 协调 Agent 由模型决定委派对象 | 开放意图与专家协作 | 前章 `sub_agents` 委派 |

> 💡 **两个不同的问题**：「路径用图还是代码表达」与「下一步由规则还是 LLM 决定」。可以让 LLM 给出计划，再由 Python 校验并执行；也可以让整个动态工作流完全不调用 LLM。动态不要求运行时创建新的 Agent。

```mermaid
flowchart TD
    U["任务输入"] --> W["Workflow 入口"]
    W --> O["@node 编排函数<br/>rerun_on_resume=True"]
    O --> A["await ctx.run_node<br/>生成内容"]
    A --> C{"Python 判断检查结果"}
    C -->|通过| F["返回成品"]
    C -->|未通过且仍有预算| R["调用修改节点"]
    R --> C
    C -->|达到上限| D["明确返回未达标状态"]
    style O fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

**注意**：Graph workflows 本身也能表示分支、回边和并行。「动态」是另一种控制流表达方式，并不意味着图只能走直线。图与代码可以在同一 `Workflow` 中组合。


---

## 2. 准备环境与事件观察器

沿用前章 `Runner + InMemorySessionService`，只是根对象换为 `Workflow`。本章不会在单元格中安装或升级依赖：从项目根目录执行 `uv sync`，再运行 `uv run --env-file .env jupyter lab` 即可。

ADK 2.x 有两个值得先记住的细节：

1. `Runner` 的新消息是 **`types.Content`**；首节点收到的不是自动解包后的字符串。下面的 `input_text()` 明确处理这个边界。
2. 工作流函数的值在 **`Event.output`** 中。多个子节点都可能产生「final response」事件；不能只凭 `is_final_response()` 判断整条流水线结束。用 `node_info.path` 识别执行位置，`node_info.output_for` 识别输出归属。

> 🔍 一个 `Workflow` 下多个函数事件的 `author` 可能都显示根工作流名称；观察调用链应看 `node_info.path`。这补充了第 4 章只看作者的观察方法。


In [1]:
import asyncio
import importlib.metadata as im
import json
import os
from time import perf_counter
from typing import Any
from uuid import uuid4

from google.adk import Context, Workflow
from google.adk.agents import Agent
from google.adk.events import RequestInput
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import NodeTimeoutError, RetryConfig, node
from google.genai import types
from pydantic import BaseModel, Field

APP, USER = "adk_ch07", "student"
print("google-adk:", im.version("google-adk"))
print("DeepSeek:", "已配置" if os.getenv("DEEPSEEK_API_KEY") else "未配置，跳过 LLM 调用")


def ds_model():
    """每个 Agent 独立接入 DeepSeek，并启用其支持的 JSON Output。"""
    return LiteLlm(
        model="deepseek/deepseek-chat",
        response_format={"type": "json_object"},
        max_tokens=1024,
    )


def input_text(value: Any) -> str:
    """将 Runner 的 Content 入口或已解包字符串统一为文本。"""
    if isinstance(value, types.Content):
        return "".join(part.text or "" for part in value.parts or [])
    if isinstance(value, str):
        return value
    raise TypeError(f"期望 Content 或 str，实际为 {type(value).__name__}")


async def collect_run(runner, session_id, *, new_message=None,
                      invocation_id=None, show_events=True):
    """消费事件流，只把归属于根 Workflow 的 output 当作本次结果。"""
    output, events = None, []
    async for event in runner.run_async(
        user_id=USER, session_id=session_id,
        new_message=new_message, invocation_id=invocation_id,
    ):
        events.append(event)
        info = event.node_info
        if show_events:
            where = info.path if info else event.author
            if event.output is not None:
                print(f"  📦 [{where}] {str(event.output)[:180]}")
            for part in event.content.parts if event.content else []:
                if part.function_call:
                    print(f"  ⚡ [{where}] {part.function_call.name}")
                elif part.text:
                    print(f"  🗣️ [{where}] {part.text[:120]}")
        if info and event.output is not None and any(
            "/" not in target and target.startswith(f"{runner.agent.name}@")
            for target in info.output_for or []
        ):
            output = event.output
    return output, events


async def run_workflow(workflow, query, *, session_id=None, show_events=True):
    """每个独立演示使用新 SessionService；返回运行时以便稍后恢复。"""
    session_id = session_id or f"demo_{uuid4().hex[:8]}"
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=workflow, app_name=APP, session_service=ss)
    message = types.Content(role="user", parts=[types.Part(text=query)])
    output, events = await collect_run(
        runner, session_id, new_message=message, show_events=show_events,
    )
    return output, events, runner, ss


print("✅ 工作流运行环境就绪")


google-adk: 2.6.3
DeepSeek: 已配置
✅ 工作流运行环境就绪


---

## 3. 最小工作流：两次节点调用就是一条流水线

三个构件各司其职：

| 构件 | 职责 | 教学类比 |
|---|---|---|
| `@node` | 把 Python 函数包装为可被调度和记录的节点 | 一个登记过的工作站 |
| `Context.run_node()` | 安排子节点执行并取得结果 | 下工单并等交付 |
| `Workflow(edges=[("START", orchestrator)])` | 连接 Runner 入口与编排函数 | 开启这一条工单 |

编排节点必须设置 **`rerun_on_resume=True`**；本地 2.6.3 会对缺失配置直接报错，而非等到发生暂停才出问题。叶子节点不调度其他节点，默认保持 `False`。

> 📌 **参数命名是接口的一部分**：默认参数绑定模式下，`ctx` 注入 `Context`，`node_input` 接收本次节点输入。把 `node_input` 随手改名为 `request`，框架会尝试从 `ctx.state["request"]` 取值，不是位置参数调用。第 4 节会介绍显式字段绑定。


In [2]:
@node
def normalize_topic(node_input: str) -> str:
    return node_input.strip().replace("主题：", "", 1)


@node
def make_heading(node_input: str) -> str:
    return f"学习主题：{node_input}"


@node(rerun_on_resume=True)
async def hello_workflow(ctx: Context, node_input):
    topic = await ctx.run_node(normalize_topic, input_text(node_input))
    heading = await ctx.run_node(make_heading, topic)
    return heading


hello = Workflow(name="hello", edges=[("START", hello_workflow)])
result, *_ = await run_workflow(hello, "  主题：动态工作流  ")
print("🏁 工作流结果：", result)
assert result == "学习主题：动态工作流"


  📦 [hello@1/hello_workflow@1/normalize_topic@1] 动态工作流
  📦 [hello@1/hello_workflow@1/make_heading@1] 学习主题：动态工作流
  📦 [hello@1/hello_workflow@1] 学习主题：动态工作流
🏁 工作流结果： 学习主题：动态工作流


> 🔍 路径依次出现 `hello_workflow / normalize_topic` 与 `hello_workflow / make_heading`。每个 `await` 等上一步完成，返回值通过局部变量直接交给下一步；不用先把每个值都写入 State。

这与直接执行一个未包装的普通函数不同：**通过 `ctx.run_node()` 才会纳入节点调度、事件、重试与 checkpoint**。普通计算可以留在函数内，但需要独立恢复和治理的操作应切成子节点。


---

## 4. 数据传递：局部变量、Pydantic 与 Session State

### 4.1 用结构化值表达业务契约

下面仍沿用「内容生产」场景：将输入解析为 `WritingRequest`，再生成一份草稿。函数节点会依据 `node_input` 的类型注解校验/转换输入；Pydantic 返回值会被转为字典输出。**因此，拿到子节点结果后显式 `model_validate()`，不要假定返回值一定是 BaseModel 实例。**

约束是业务规则，不是给模型看的装饰：`min_length` 拦住空主题，`Literal` 将风格限制在已支持的集合。


In [3]:
from typing import Literal


class WritingRequest(BaseModel):
    topic: str = Field(min_length=1)
    style: Literal["brief", "detailed"] = "brief"


class Draft(BaseModel):
    title: str
    body: str


@node
def parse_request(node_input: str) -> WritingRequest:
    return WritingRequest.model_validate_json(node_input)


@node
def draft_from_request(node_input: WritingRequest) -> Draft:
    body = f"先理解{node_input.topic}的核心概念，再运行一个小例子。"
    return Draft(title=node_input.topic, body=body)


@node(rerun_on_resume=True)
async def typed_workflow(ctx: Context, node_input):
    request = await ctx.run_node(parse_request, input_text(node_input))
    draft = Draft.model_validate(await ctx.run_node(draft_from_request, request))
    return {"title": draft.title, "characters": len(draft.body)}


typed = Workflow(name="typed", edges=[("START", typed_workflow)])
query = WritingRequest(topic="ADK 动态编排").model_dump_json()
result, *_ = await run_workflow(typed, query)
print(result)


  📦 [typed@1/typed_workflow@1/parse_request@1] {'topic': 'ADK 动态编排', 'style': 'brief'}
  📦 [typed@1/typed_workflow@1/draft_from_request@1] {'title': 'ADK 动态编排', 'body': '先理解ADK 动态编排的核心概念，再运行一个小例子。'}
  📦 [typed@1/typed_workflow@1] {'title': 'ADK 动态编排', 'characters': 26}
{'title': 'ADK 动态编排', 'characters': 26}


### 4.2 默认参数绑定与显式字段绑定

`@node` 的默认 `parameter_binding="state"` 会特殊处理 `ctx` 与 `node_input`，其他具名参数取自 State。若想把一个输入字典解包为多个函数参数，使用 **`parameter_binding="node_input"`**。


In [4]:
@node(parameter_binding="node_input")
def make_label(topic: str, style: str = "brief") -> str:
    return f"{topic} / {style}"


@node(rerun_on_resume=True)
async def binding_workflow(ctx: Context, node_input):
    request = WritingRequest.model_validate_json(input_text(node_input))
    return await ctx.run_node(make_label, request.model_dump())


binding = Workflow(name="binding", edges=[("START", binding_workflow)])
result, *_ = await run_workflow(binding, query)
assert result == "ADK 动态编排 / brief"


  📦 [binding@1/binding_workflow@1/make_label@1] ADK 动态编排 / brief
  📦 [binding@1/binding_workflow@1] ADK 动态编排 / brief


### 4.3 State 什么时候仍然必要？

| 数据 | 放在哪里 | 原因 |
|---|---|---|
| 某步输出与下一步输入 | 返回值 + 局部变量 | 依赖明确，减少键名耦合 |
| 工具/回调需要共享的摘要、流程状态 | `ctx.state` | 让不同组件读取 |
| Agent instruction 的 `{draft}` 插值 | State 中的 `draft` | 沿用第 2、4 章的模板方式 |
| 大报告、图片或文件 | Artifact / 外部存储 | 避免膨胀会话事件 |

在函数节点中写 **`ctx.state[...]`**，框架会把变更附加到事件中；不要绕过事件机制直接改 Session 对象。Python 局部变量本身不是持久化数据库：恢复时依赖「重新进入编排函数 + 回放已完成子节点结果」重建它们。

参考：[官方数据传递与 schema](https://adk.dev/graphs/data-handling/)。


---

## 5. 条件与迭代：让运行结果决定下一步

### 5.1 条件分支：短版与长版走不同节点

`if` 负责选择，`ctx.run_node()` 负责执行。两个候选节点可以提前定义好；动态的是本次选哪个，并不要求不停创建 Agent。


In [5]:
@node
def brief_outline(node_input: WritingRequest) -> list[str]:
    return [f"{node_input.topic}是什么", "最小示例"]


@node
def detailed_outline(node_input: WritingRequest) -> list[str]:
    return [f"{node_input.topic}是什么", "最小示例", "高级功能", "常见错误"]


@node(rerun_on_resume=True)
async def outline_workflow(ctx: Context, node_input):
    request = WritingRequest.model_validate_json(input_text(node_input))
    selected = brief_outline if request.style == "brief" else detailed_outline
    return await ctx.run_node(selected, request.model_dump())


outline = Workflow(name="outline", edges=[("START", outline_workflow)])
for style, expected in [("brief", 2), ("detailed", 4)]:
    result, *_ = await run_workflow(
        outline, WritingRequest(topic="动态工作流", style=style).model_dump_json(),
    )
    assert len(result) == expected


  📦 [outline@1/outline_workflow@1/brief_outline@1] ['动态工作流是什么', '最小示例']
  📦 [outline@1/outline_workflow@1] ['动态工作流是什么', '最小示例']
  📦 [outline@1/outline_workflow@1/detailed_outline@1] ['动态工作流是什么', '最小示例', '高级功能', '常见错误']
  📦 [outline@1/outline_workflow@1] ['动态工作流是什么', '最小示例', '高级功能', '常见错误']


### 5.2 有上限的迭代：业务不达标不是运行异常

模拟「草稿 → 规则检查 → 修订」。检查器要求包含 **定义 / 示例 / 边界**三个词；修订器每次补一项。这不是内容质量评测，只是让循环行为可复现。

与第 4 章的 `LoopAgent + escalate` 相比，现在退出条件直接写在 Python 中。约定 **`max_revisions` 是修改次数**，不是包括初检在内的检查次数；返回 `passed` 与 `stop_reason`，避免达到上限后仍把草稿标成成功。


In [6]:
@node
def check_draft(node_input: str) -> dict:
    missing = [word for word in ("定义", "示例", "边界") if word not in node_input]
    return {"passed": not missing, "missing": missing}


@node
def revise_draft(node_input: dict) -> str:
    return node_input["draft"] + "；" + node_input["missing"][0]


@node(rerun_on_resume=True)
async def refinement(ctx: Context, node_input: dict) -> dict:
    draft = node_input["draft"]
    max_revisions = node_input["max_revisions"]
    if max_revisions < 0:
        raise ValueError("max_revisions 不能为负数")
    for revision in range(max_revisions + 1):
        review = await ctx.run_node(check_draft, draft)
        if review["passed"]:
            return {"draft": draft, "passed": True,
                    "revisions": revision, "stop_reason": "passed"}
        if revision == max_revisions:
            return {"draft": draft, "passed": False, "revisions": revision,
                    "stop_reason": "revision_limit", "missing": review["missing"]}
        draft = await ctx.run_node(revise_draft, {"draft": draft, **review})


@node(rerun_on_resume=True)
async def refinement_entry(ctx: Context, node_input):
    return await ctx.run_node(refinement, json.loads(input_text(node_input)))


refiner = Workflow(name="refiner", edges=[("START", refinement_entry)])
for draft, budget, passed in [("定义、示例、边界", 0, True),
                              ("定义", 2, True), ("定义", 1, False)]:
    result, *_ = await run_workflow(
        refiner, json.dumps({"draft": draft, "max_revisions": budget}),
    )
    print("🏁", result)
    assert result["passed"] is passed


  📦 [refiner@1/refinement_entry@1/refinement@1/check_draft@1] {'passed': True, 'missing': []}
  📦 [refiner@1/refinement_entry@1/refinement@1] {'draft': '定义、示例、边界', 'passed': True, 'revisions': 0, 'stop_reason': 'passed'}
  📦 [refiner@1/refinement_entry@1] {'draft': '定义、示例、边界', 'passed': True, 'revisions': 0, 'stop_reason': 'passed'}
🏁 {'draft': '定义、示例、边界', 'passed': True, 'revisions': 0, 'stop_reason': 'passed'}
  📦 [refiner@1/refinement_entry@1/refinement@1/check_draft@1] {'passed': False, 'missing': ['示例', '边界']}
  📦 [refiner@1/refinement_entry@1/refinement@1/revise_draft@1] 定义；示例
  📦 [refiner@1/refinement_entry@1/refinement@1/check_draft@2] {'passed': False, 'missing': ['边界']}
  📦 [refiner@1/refinement_entry@1/refinement@1/revise_draft@2] 定义；示例；边界
  📦 [refiner@1/refinement_entry@1/refinement@1/check_draft@3] {'passed': True, 'missing': []}
  📦 [refiner@1/refinement_entry@1/refinement@1] {'draft': '定义；示例；边界', 'passed': True, 'revisions': 2, 'stop_reason': 'passed'}
  📦 [refiner@1/ref

> 🔍 三种运行分别覆盖一次通过、修改后通过与预算耗尽。**循环次数依据结果变化**，调度方式仍受代码约束。业务评审失败应触发修订分支；网络异常才交给第 9 节的 RetryConfig，二者不应混为一谈。


---

## 6. 接回 DeepSeek：Agent 与函数共用一套节点接口

前面的函数节点换成 `Agent`，编排代码依然是 `await ctx.run_node(agent, input)`。场景继续使用内容流水线：**生成初稿 → 本地规则检查 → 必要时让模型修改一次 → 再检查**。

沿用第 2 章的 DeepSeek 兼容方式：**JSON Output + 本地 Pydantic 校验**。`ds_model()` 显式发送 `response_format={"type": "json_object"}`；Agent 不设置 `output_schema=Draft`，避免适配层将它转换为 DeepSeek 不支持的 `json_schema` 请求。

Agent 返回 JSON 字符串，使用 `Draft.model_validate_json()` 解析并校验 title、body 字段，再交给函数节点。JSON Output 约束的是 JSON 格式，不保证字段、事实或字数；字段由 Pydantic 校验，字数与空值由本地规则检查。参考：[DeepSeek 官方 JSON Output](https://api-docs.deepseek.com/guides/json_mode/)。

下面将 Agent 设置为 **`mode="single_turn"`**，任务完成后把结果交回编排函数，不让工作节点接管用户对话。本例使用显式输入和静态 instruction，避免为了简单传值维护大量 `output_key`。


In [7]:
writer = Agent(
    name="dynamic_writer", model=ds_model(), mode="single_turn",
    instruction="你是中文技术写作老师。根据输入写一段 60～120 字的入门介绍，"
                "包含一个实际例子。只输出一个 JSON 对象，title 和 body 均为字符串；"
                "title 是标题，body 是正文，不要 Markdown 代码块或其他文字。",
    description="按主题写一段结构化技术介绍",
)
reviser = Agent(
    name="dynamic_reviser", model=ds_model(), mode="single_turn",
    instruction="根据输入的 draft 和 findings 修改文章，正文控制在 60～120 字，"
                "保留实际例子。只输出一个 JSON 对象，title 和 body 均为字符串；"
                "title 是标题，body 是正文，不要 Markdown 代码块或其他文字。",
    description="根据明确检查意见修改文章",
)


@node
def check_article(node_input: Draft) -> dict:
    findings = []
    if not node_input.title.strip():
        findings.append("标题不能为空")
    if not 60 <= len(node_input.body) <= 120:
        findings.append(f"正文有 {len(node_input.body)} 字，要求 60～120 字")
    return {"passed": not findings, "findings": findings}


@node(rerun_on_resume=True)
async def article_workflow(ctx: Context, node_input):
    draft = Draft.model_validate_json(
        await ctx.run_node(writer, input_text(node_input)),
    )
    review = await ctx.run_node(check_article, draft.model_dump())
    revisions = 0
    if not review["passed"]:
        draft = Draft.model_validate_json(await ctx.run_node(
            reviser, {"draft": draft.model_dump(), "findings": review["findings"]},
        ))
        review = await ctx.run_node(check_article, draft.model_dump())
        revisions = 1
    # 并发复用这个流程时，每个执行路径的摘要使用独立键，避免覆盖。
    ctx.state[f"article_summary:{ctx.node_path}"] = {"title": draft.title, **review}
    return {"draft": draft.model_dump(), "revisions": revisions, **review}


article = Workflow(name="article", edges=[("START", article_workflow)])
if os.getenv("DEEPSEEK_API_KEY"):
    result, *_ = await run_workflow(article, "介绍 ADK Dynamic workflows")
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("⏭️ 已定义 Agent；配置 DEEPSEEK_API_KEY 后可运行模型示例。")


  🗣️ [article@1/article_workflow@1/dynamic_writer@1] {"title": "ADK 动态工作流入门", "body": "ADK 动态工作流允许你在运行时根据任务状态灵活编排 Agent 步骤，而非预先写死流程。典型场景是客服机器人：用户先问退货政策，系统识别意图后自动调用政策查询 Agent
  📦 [article@1/article_workflow@1/check_article@1] {'passed': False, 'findings': ['正文有 146 字，要求 60～120 字']}
  🗣️ [article@1/article_workflow@1/dynamic_reviser@1] {"title":"ADK 动态工作流入门","body":"ADK 动态工作流可在运行时按状态编排 Agent。客服场景：用户问退货政策，识别意图后调用政策查询 Agent；若要求退货，再插入订单核验与退款 Agent，像搭积木按需组合。
  📦 [article@1/article_workflow@1/check_article@2] {'passed': True, 'findings': []}
  📦 [article@1/article_workflow@1] {'draft': {'title': 'ADK 动态工作流入门', 'body': 'ADK 动态工作流可在运行时按状态编排 Agent。客服场景：用户问退货政策，识别意图后调用政策查询 Agent；若要求退货，再插入订单核验与退款 Agent，像搭积木按需组合。'}, 'revisions': 1, 'passed': True, 'findings':
{
  "draft": {
    "title": "ADK 动态工作流入门",
    "body": "ADK 动态工作流可在运行时按状态编排 Agent。客服场景：用户问退货政策，识别意图后调用政策查询 Agent；若要求退货，再插入订单核验与退款 Agent，像搭积木按需组合。"
  },
  "revisions": 1,
  "passed": true,
  "findings": []
}


> 🔍 注意 `Event.output` 的类型取决于节点：本例未启用 Agent 的 `output_schema`，所以 `ctx.run_node(writer, ...)` 返回 JSON **字符串**，经 `Draft.model_validate_json()` 后才得到类型明确的对象；后续函数节点返回的是字典。JSON 格式错误或字段类型错误会直接触发校验异常，不能伪装成合格文章。修改一次仍不达标时会明确返回 `passed=False`，供上层决定再审、人工处理或终止。摘要使用执行路径区分 State key，批量复用时不会互相覆盖。
>
> `ctx.run_node()` 是「调用—返回」；它不是第 4 章 `transfer_to_agent` 的「移交—接管」。当业务已经决定下一步执行谁时，无需再加一个协调 LLM 来重新决定。


---

## 7. 动态并发：输入有几项，就扇出几项

### 7.1 `asyncio.gather`：先安排，再一起等待

模拟对每个主题提取元数据。`asyncio.sleep()` 代表异步 I/O；真正的 CPU 密集计算不会因为加了 `async` 就自动并行。

下面先生成 `ctx.run_node()` 协程列表，再一次 `await asyncio.gather(...)`。如果在 `for` 中逐个 `await`，得到的是串行。**结果按传入顺序返回，完成顺序可以不同。**


In [8]:
@node
async def inspect_topic(node_input: str) -> dict:
    await asyncio.sleep(0.03)
    return {"topic": node_input, "characters": len(node_input)}


@node(rerun_on_resume=True)
async def fanout(ctx: Context, node_input):
    topics = json.loads(input_text(node_input))
    tasks = [ctx.run_node(inspect_topic, topic) for topic in topics]
    return await asyncio.gather(*tasks)


fanout_app = Workflow(name="fanout_app", edges=[("START", fanout)])
topics = ["节点", "循环", "checkpoint", "人工确认"]
start = perf_counter()
result, *_ = await run_workflow(fanout_app, json.dumps(topics), show_events=False)
print("结果：", result)
print(f"并发总耗时：{perf_counter() - start:.3f}s（包含框架调度，不是性能基准）")
assert [item["topic"] for item in result] == topics


结果： [{'topic': '节点', 'characters': 2}, {'topic': '循环', 'characters': 2}, {'topic': 'checkpoint', 'characters': 10}, {'topic': '人工确认', 'characters': 4}]
并发总耗时：0.041s（包含框架调度，不是性能基准）


### 7.2 常用高级功能：内置 parallel worker 与并发上限

批量调用同一种节点时，2.6.3 支持 **`node(worker, parallel_worker=True, max_parallel_workers=2)`**。它把列表输入展开，最多同时运行两个子节点，最后按输入顺序汇聚结果；`node()` 除了作为装饰器，也可以包装已定义的节点。

比起把限流逻辑、`create_task()` 和恢复语义混在一起，内置 worker 更适合常规 Map 场景。手工 `gather` 适合不同节点的并行组合。不要让 `ctx.run_node()` 成为无人等待的后台任务。


In [9]:
bounded_worker = node(
    inspect_topic, parallel_worker=True, max_parallel_workers=2,
)


@node(rerun_on_resume=True)
async def bounded_fanout(ctx: Context, node_input):
    topics = json.loads(input_text(node_input))
    return await ctx.run_node(bounded_worker, topics)


bounded = Workflow(name="bounded", edges=[("START", bounded_fanout)])
result, *_ = await run_workflow(bounded, json.dumps(topics), show_events=False)
assert [item["topic"] for item in result] == topics
print("✅ 带并发上限的批处理：", result)


✅ 带并发上限的批处理： [{'topic': '节点', 'characters': 2}, {'topic': '循环', 'characters': 2}, {'topic': 'checkpoint', 'characters': 10}, {'topic': '人工确认', 'characters': 4}]


### 7.3 并发故障与共享状态怎么处理？

1. **共享 State 的同一键不是自动合并器**：两个分支都写 `state["result"]` 可能互相覆盖。优先返回各自结果，父节点统一汇总；若必须写 State，使用不同键或按业务设计合并。
2. **部分成功也需要定义语义**：是整体失败，还是返回成功项与失败项？`gather` 默认会传播异常，不能把缺失项当空字符串。`return_exceptions=True` 也不能把 HITL 暂停信号误当普通失败吞掉。
3. **并发预算分层设置**：内置 worker 限制的是这一批节点，不等于全服务、全租户的 API 限流。生产系统仍需要整体配额。
4. **恢复需要稳定的安排顺序**：同一批任务应有确定排序，不要按网络返回顺序生成执行 ID。先读取一个带 checkpoint 的列表快照，再扇出通常更容易保证一致性。

第 12 节会使用内置 worker 做一个有上限的批量 Agent 示例。


---

## 8. 组合与封装：动态节点可以调用另一个动态节点

第 5 节的 `refinement` 已是一段可复用的业务流程。把它放到批处理外层，就得到「多个主题，每个主题内部自己修订」的两层结构。**每个调用子节点的层都要开启 `rerun_on_resume=True`。**

```mermaid
flowchart LR
    E["外层编排"] --> R1["refinement：主题 A"]
    E --> R2["refinement：主题 B"]
    R1 --> C1["check / revise"]
    R2 --> C2["check / revise"]
```

不需要引入新的抽象类。把有独立输入输出、预算和恢复边界的业务逻辑封装为节点，就能复用和观察。


In [10]:
@node(rerun_on_resume=True)
async def nested_workflow(ctx: Context, node_input):
    drafts = json.loads(input_text(node_input))
    outputs = []
    for draft in drafts:
        outputs.append(await ctx.run_node(
            refinement, {"draft": draft, "max_revisions": 2},
        ))
    return outputs


nested = Workflow(name="nested", edges=[("START", nested_workflow)])
result, *_ = await run_workflow(nested, json.dumps(["定义", "定义、示例、边界"]))
assert all(item["passed"] for item in result)


  📦 [nested@1/nested_workflow@1/refinement@1/check_draft@1] {'passed': False, 'missing': ['示例', '边界']}
  📦 [nested@1/nested_workflow@1/refinement@1/revise_draft@1] 定义；示例
  📦 [nested@1/nested_workflow@1/refinement@1/check_draft@2] {'passed': False, 'missing': ['边界']}
  📦 [nested@1/nested_workflow@1/refinement@1/revise_draft@2] 定义；示例；边界
  📦 [nested@1/nested_workflow@1/refinement@1/check_draft@3] {'passed': True, 'missing': []}
  📦 [nested@1/nested_workflow@1/refinement@1] {'draft': '定义；示例；边界', 'passed': True, 'revisions': 2, 'stop_reason': 'passed'}
  📦 [nested@1/nested_workflow@1/refinement@2/check_draft@1] {'passed': True, 'missing': []}
  📦 [nested@1/nested_workflow@1/refinement@2] {'draft': '定义、示例、边界', 'passed': True, 'revisions': 0, 'stop_reason': 'passed'}
  📦 [nested@1/nested_workflow@1] [{'draft': '定义；示例；边界', 'passed': True, 'revisions': 2, 'stop_reason': 'passed'}, {'draft': '定义、示例、边界', 'passed': True, 'revisions': 0, 'stop_reason': 'passed'}]


### 8.1 图与动态节点混合

图式 `edges` 可以串起「入口解析 → 动态分发 → 最终格式化」。`Workflow` 本身也可以经 `ctx.run_node()` 作为子流程调用，而不必把所有业务展开到一个巨大函数中。

> ⚠️ **2.6.3 的边界**：`Workflow` 不能直接当作 `LlmAgent.sub_agents` 的元素。前章所说工作流 Agent 可嵌套的场景指旧的 `Sequential / Parallel / Loop` Agent，不应直接套到新 `Workflow`。本章把 `Workflow` 用作 Runner 根节点，把 Agent 用作它的工作节点。

递归也是普通 Python 控制流，但要显式限制深度/总节点数；通常先用有上限的迭代与嵌套更容易排查。下面先运行图与动态节点混合，再看递归。


In [11]:
@node
def topic_list(node_input) -> list[str]:
    return json.loads(input_text(node_input))


@node(rerun_on_resume=True)
async def inspect_many(ctx: Context, node_input: list[str]) -> list[dict]:
    return await ctx.run_node(bounded_worker, node_input)


@node
def format_many(node_input: list[dict]) -> str:
    return "\n".join(f"- {item['topic']}：{item['characters']} 字" for item in node_input)


hybrid = Workflow(
    name="hybrid", edges=[("START", topic_list, inspect_many, format_many)],
)
result, *_ = await run_workflow(hybrid, json.dumps(topics), show_events=False)
print(result)


- 节点：2 字
- 循环：2 字
- checkpoint：10 字
- 人工确认：4 字


### 8.2 有界递归：处理运行时才知道层数的任务树

例如课程目录中的小节还包含小节，不必为每一层手写边。一个动态节点可以安排自己的下一层执行，节点路径会记录层级。本例限制树深度为 3，每层最多 2 个孩子；从根深度 0 开始，最多处理 15 个节点。

实际调用的是 `ctx.run_node(count_sections, ...)`，所以每一层仍有独立执行记录；不是绕过框架直接递归调用底层 Python 函数。


In [12]:
@node(rerun_on_resume=True)
async def count_sections(ctx: Context, node_input: dict) -> dict:
    depth = node_input.get("depth", 0)
    children = node_input.get("children", [])
    if depth > 3 or len(children) > 2:
        raise ValueError("目录超过允许的深度或分支数量")
    total = 1
    for child in children:
        result = await ctx.run_node(
            count_sections, {**child, "depth": depth + 1},
        )
        total += result["count"]
    return {"title": node_input["title"], "count": total}


@node(rerun_on_resume=True)
async def tree_entry(ctx: Context, node_input):
    tree = json.loads(input_text(node_input))
    return await ctx.run_node(count_sections, {**tree, "depth": 0})


tree_app = Workflow(name="tree_app", edges=[("START", tree_entry)])
tree = {"title": "动态工作流", "children": [
    {"title": "基础", "children": [{"title": "节点"}]},
    {"title": "恢复"},
]}
result, *_ = await run_workflow(tree_app, json.dumps(tree), show_events=False)
assert result["count"] == 4
print("✅ 目录递归计数：", result)


✅ 目录递归计数： {'title': '动态工作流', 'count': 4}


---

## 9. 重试、超时与降级：节点的运行保障

### 9.1 `RetryConfig`：只重试有希望恢复的错误

把远程读取模拟成「第一次连接失败，第二次成功」。重试配置放在叶子节点上，让框架知道失败发生在哪一步。

| 参数 | 本章设置 | 含义 |
|---|---|---|
| `max_attempts` | `3` | **包括首次请求**，最多 3 次尝试 |
| `initial_delay / max_delay` | `0.01 / 0.05` | 重试间隔与上限（秒） |
| `backoff_factor` | `2` | 间隔按倍率增加 |
| `jitter` | `0` | 演示关闭随机抖动；生产常用于分散重试 |
| `exceptions` | `[ConnectionError]` | 仅对连接异常重试 |

生产中输入校验失败、业务拒绝通常不该重试；发送订单等操作只有具备业务幂等性才能安全重试。不要在工具内部 `except Exception: return "失败"`，那会把失败伪装成成功，框架无法按 RetryConfig 处理。

> 🔍 本节故意制造一次错误，ADK 会打印失败日志，再按配置重试；第 9.2 节也会打印模拟超时日志。看到日志后继续取得预期结果是演示的一部分。2.6.3 的局部重试计数不会跨恢复持久化：多次恢复的总尝试预算还需要业务层控制。


In [13]:
network_calls = {"count": 0}  # 仅作演示计数；不是生产中的持久化状态


@node(retry_config=RetryConfig(
    max_attempts=3, initial_delay=0.01, max_delay=0.05,
    backoff_factor=2, jitter=0, exceptions=[ConnectionError],
))
async def flaky_read(ctx: Context, node_input: str) -> dict:
    network_calls["count"] += 1
    print("远程读取 attempt_count =", ctx.attempt_count)
    if network_calls["count"] == 1:
        raise ConnectionError("模拟一次临时断连")
    return {"topic": node_input, "source": "remote"}


@node(rerun_on_resume=True)
async def retry_workflow(ctx: Context, node_input):
    return await ctx.run_node(flaky_read, input_text(node_input))


retry_app = Workflow(name="retry_app", edges=[("START", retry_workflow)])
network_calls["count"] = 0
result, *_ = await run_workflow(retry_app, "动态工作流", show_events=False)
assert network_calls["count"] == 2
print("✅ 第二次成功：", result)


远程读取 attempt_count =

Node execution failed with exception
Traceback (most recent call last):
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_base_node.py", line 166, in run
    async for item in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_function_node.py", line 538, in _run_impl
    result = await self._func(**kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\37245\AppData\Local\Temp\ipykernel_45136\1729208710.py", line 12, in flaky_read
    raise ConnectionError("模拟一次临时断连")
ConnectionError: 模拟一次临时断连
Node flaky_read failed and is being retried locally. No

 1
远程读取 attempt_count = 2
✅ 第二次成功： {'topic': '动态工作流', 'source': 'remote'}


### 9.2 `timeout` 与局部降级

`timeout` 的单位是秒。下面模拟一个超过节点预算的读取，并在父节点使用**明确的超时降级**。2.6.3 的动态子节点失败会被包装成异常；因此查看其 `error`，仅当根因是 `NodeTimeoutError` 才返回本地兜底，其他异常继续抛出。

> 💡 超时限制的是该节点的等待预算，不是整条工作流的 SLA，也不保证外部系统已撤销副作用。同步阻塞代码应改为异步 I/O 或设计专门的执行隔离；不要在 async 节点中用 `time.sleep()`。


In [14]:
@node(timeout=0.01, retry_config=RetryConfig(max_attempts=1))
async def slow_read(node_input: str) -> dict:
    await asyncio.sleep(0.1)
    return {"topic": node_input, "source": "remote"}


@node(rerun_on_resume=True)
async def timeout_workflow(ctx: Context, node_input):
    topic = input_text(node_input)
    try:
        return await ctx.run_node(slow_read, topic)
    except Exception as exc:
        cause = getattr(exc, "error", exc)
        if not isinstance(cause, NodeTimeoutError):
            raise
        return {"topic": topic, "source": "local", "degraded": True}


timeout_app = Workflow(name="timeout_app", edges=[("START", timeout_workflow)])
result, *_ = await run_workflow(timeout_app, "动态工作流", show_events=False)
assert result["degraded"] is True
print("✅ 超时后明确降级：", result)


Node execution failed with exception
Traceback (most recent call last):
  File "D:\Python\Lib\asyncio\tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "D:\Python\Lib\site-packages\google\adk\workflow\_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_base_node.py", line 166, in run
    async for item in agen:
  File "D:\Python\Lib\site-packages\google\adk\workflow\_function_node.py", line 538, in _run_impl
    result = await self._func(**kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\37245\AppData\Local\Temp\ipykernel_45136\2433590649.py", line 3, in slow_read
    await asyncio.sleep(0.1)
  File "D:\Python\Lib\asyncio\tasks.py", line 665, in sleep
    return await future
           ^^^^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "D:\Python\Lib\si

✅ 超时后明确降级： {'topic': '动态工作流', 'source': 'local', 'degraded': True}


---

## 10. 人工确认与 checkpoint：暂停后如何接着做？

### 10.1 不是「在函数里面阻塞 input()」

人工输入节点返回或 `yield` **`RequestInput`**，工作流发出请求并暂停。本章演示「生成候选稿 → 等人工确认 → 模拟发布/拒绝」，不会执行真实发布动作。

恢复时有两种不同配置：

| 节点 | 配置 | 恢复时发生什么 |
|---|---|---|
| 编排父节点 | `rerun_on_resume=True` | 从头进入函数，取回已完成子节点结果，继续决策 |
| 简单人工输入叶子 | `rerun_on_resume=False` | 不重新执行叶子函数，把人工响应作为该步结果 |

```mermaid
sequenceDiagram
    participant R as Runner
    participant P as 编排节点
    participant C as 生成节点
    participant H as 人工输入节点
    R->>P: 首次运行
    P->>C: 生成候选稿
    C-->>P: 结果写入 checkpoint
    P->>H: RequestInput
    H-->>R: 请求确认并暂停
    R->>P: 同一 invocation 恢复
    P->>C: 再次安排同一步
    C-->>P: 回放已完成结果，函数不再执行
    P->>H: 取得人工响应
    H-->>P: approved=true/false
    P-->>R: 继续发布或拒绝
```

这里的「从头」很关键：**Python 调用栈与局部变量并没有被原样冻结**。父函数会重跑，已成功子节点通常通过 checkpoint 回放；父函数内直接执行的代码也会重跑。不要把发邮件等副作用直接写在编排函数顶部。

参考：[官方人工输入](https://adk.dev/graphs/human-input/) · [动态工作流的人工输入](https://adk.dev/graphs/dynamic/#human-input)。


In [15]:
approval_counts = {"parent": 0, "prepare": 0, "publish": 0}


class ApprovalDecision(BaseModel):
    approved: bool


@node
def prepare_candidate(node_input: str) -> dict:
    approval_counts["prepare"] += 1
    return {"topic": node_input, "body": "定义 → 示例 → 边界"}


@node(rerun_on_resume=False)
def ask_approval(node_input: dict):
    return RequestInput(
        message="是否接受这份候选稿？请返回 approved=true 或 false。",
        payload=node_input, response_schema=ApprovalDecision,
    )


@node
def publish_candidate(node_input: dict) -> dict:
    approval_counts["publish"] += 1
    return {"status": "published_demo", "draft": node_input}


@node(rerun_on_resume=True)
async def approval_workflow(ctx: Context, node_input):
    approval_counts["parent"] += 1  # 用于观察父函数重新进入
    draft = await ctx.run_node(prepare_candidate, input_text(node_input))
    reply = await ctx.run_node(ask_approval, draft)
    decision = ApprovalDecision.model_validate(reply)
    if not decision.approved:
        return {"status": "rejected", "draft": draft}
    return await ctx.run_node(publish_candidate, draft)


approval_app = Workflow(name="approval_app", edges=[("START", approval_workflow)])
approval_counts.update(parent=0, prepare=0, publish=0)
paused, pause_events, approval_runner, approval_ss = await run_workflow(
    approval_app, "动态工作流介绍", session_id="approval_yes",
)
assert paused is None
assert approval_counts == {"parent": 1, "prepare": 1, "publish": 0}
print("⏸️ 已暂停，尚未产生工作流最终输出。")


  📦 [approval_app@1/approval_workflow@1/prepare_candidate@1] {'topic': '动态工作流介绍', 'body': '定义 → 示例 → 边界'}
  ⚡ [approval_app@1/approval_workflow@1/ask_approval@1] adk_request_input
⏸️ 已暂停，尚未产生工作流最终输出。


### 10.2 构造对应的 FunctionResponse，恢复同一次 invocation

从暂停事件取得请求的 **function call id** 与 **invocation_id**；回复必须匹配真实请求。恢复不是创建新会话、不是启动一个新任务，也不是仅发一条普通的「同意」文本。

下面代码**模拟**用户/前端已收到确认结果，并以 FunctionResponse 送回 Runner。真实应用应由用户界面或可信业务系统提供 `approved`，编排代码不能自行代替用户批准。

新 `Workflow` 的节点 checkpoint 随 Session 事件保存；本例无须额外添加旧工作流 Agent 的 `ResumabilityConfig`。`InMemorySessionService` 只支持当前进程内的演示：若要重启后恢复，必须保存会话/事件并重新加载兼容的流程代码。


In [16]:
def pending_input(events):
    """本例只有一个待确认请求；从真实暂停事件中取出它。"""
    for event in reversed(events):
        for part in event.content.parts if event.content else []:
            call = part.function_call
            if call and call.name == "adk_request_input":
                return event.invocation_id, call
    raise ValueError("没有找到待处理的 RequestInput")


async def answer_approval(runner, session_id, events, approved):
    invocation_id, call = pending_input(events)
    response = types.Content(role="user", parts=[types.Part(
        function_response=types.FunctionResponse(
            id=call.id, name=call.name, response={"approved": approved},
        ),
    )])
    return await collect_run(
        runner, session_id, new_message=response, invocation_id=invocation_id,
    )


result, resume_events = await answer_approval(
    approval_runner, "approval_yes", pause_events, True,
)
print("🏁", result)
print("执行计数：", approval_counts)
assert result["status"] == "published_demo"
assert approval_counts == {"parent": 2, "prepare": 1, "publish": 1}


  📦 [approval_app@1/approval_workflow@1/publish_candidate@1] {'status': 'published_demo', 'draft': {'topic': '动态工作流介绍', 'body': '定义 → 示例 → 边界'}}
  📦 [approval_app@1/approval_workflow@1] {'status': 'published_demo', 'draft': {'topic': '动态工作流介绍', 'body': '定义 → 示例 → 边界'}}
🏁 {'status': 'published_demo', 'draft': {'topic': '动态工作流介绍', 'body': '定义 → 示例 → 边界'}}
执行计数： {'parent': 2, 'prepare': 1, 'publish': 1}


> 🔍 **父函数执行了 2 次，生成函数只执行了 1 次**。第二次 `ctx.run_node(prepare_candidate, ...)` 使用已完成节点的结果；只有确认后的发布节点真正运行。这比单看结果更能证明 checkpoint 起作用。

下面再覆盖拒绝路径：即使草稿已生成，没有接受也不能进入发布节点。


In [17]:
approval_counts.update(parent=0, prepare=0, publish=0)
_, denied_events, denied_runner, _ = await run_workflow(
    approval_app, "另一份候选稿", session_id="approval_no", show_events=False,
)
result, _ = await answer_approval(denied_runner, "approval_no", denied_events, False)
assert result["status"] == "rejected"
assert approval_counts["publish"] == 0
print("✅ 拒绝后没有模拟发布：", result["status"])


  📦 [approval_app@1/approval_workflow@1] {'status': 'rejected', 'draft': {'topic': '另一份候选稿', 'body': '定义 → 示例 → 边界'}}
✅ 拒绝后没有模拟发布： rejected


### 10.3 业务失败、运行失败与暂停是三种状态

| 状态 | 正确处理 | 容易犯的错 |
|---|---|---|
| 质量不达标 | 返回检查结果，修订或人工处理 | 用无限重试掩盖模型能力不足 |
| 网络/执行异常 | 适当重试，最终传播或明确降级 | 捕获后返回普通成功值 |
| `RequestInput` 暂停 | 保留 invocation，等待对应人工响应 | 当作失败重试，或假设 `None` 表示拒绝 |

在本地 2.6.3 中，`ctx.run_node()` 会向上传播子节点的暂停；不要用 **`except BaseException`** 吞掉它，也不要用 `gather(return_exceptions=True)` 把暂停当作正常业务结果。`raise_on_wait=True` 适用于另一类「没有输出而等待」的子节点，不替代 `RequestInput` 的对应响应协议。

完成后再次用同一个 `session_id` 发普通消息，通常是新 invocation；完成结果的复用不是「同一 Session 永久缓存所有节点」。恢复应围绕同一 invocation 和匹配的节点身份展开。


---

## 11. Execution ID：恢复时框架怎么认出「同一步」？

### 11.1 三种 ID 不要混淆

| 标识 | 作用 | 示例 |
|---|---|---|
| `session_id` | 定位一份会话 | `approval_yes` |
| `invocation_id` | 定位一次工作流执行及其恢复 | 暂停事件中的 `event.invocation_id` |
| 节点 `run_id` / `node_info.path` | 定位这一轮父节点下的子节点执行 | `approval_app@1/approval_workflow@1/prepare_candidate@1` |

默认节点身份由父路径、节点名与调用计数构成，恢复时按相同身份寻找完成结果。优先用默认 ID，并保证安排顺序可复现。

**恢复一致性的设计原则**：把「取列表」「生成计划」「读取时间/外部配置」放进可 checkpoint 的叶子节点；父函数使用它们的结果做决定。不要在父函数重新进入时重新读取一个变化的外部列表，也不要每次用当前时间决定不同分支。

### 11.2 自定义 `run_id`：只在需要稳定业务身份时使用

对于允许排序变化的批次，可以给每项显式 `run_id=f"item-{id}"`。本地 API 要求含非数字字符；同一父节点中要保持唯一。**同一身份不能在恢复时代表不同输入或业务版本**，否则会错误复用旧结果。

官方建议谨慎使用自定义 ID：[Execution IDs](https://adk.dev/graphs/dynamic/#execution-ids)。下面演示显式映射，并首先检查业务 ID 不重复。


In [18]:
@node
def process_item(node_input: dict) -> dict:
    return {"id": node_input["id"], "label": node_input["topic"].upper()}


@node(rerun_on_resume=True)
async def stable_batch(ctx: Context, node_input):
    items = json.loads(input_text(node_input))
    ids = [item["id"] for item in items]
    if len(ids) != len(set(ids)):
        raise ValueError("同一批次的业务 ID 必须唯一")
    tasks = [ctx.run_node(process_item, item, run_id=f"item-{item['id']}")
             for item in items]
    return await asyncio.gather(*tasks)


stable = Workflow(name="stable", edges=[("START", stable_batch)])
items = [{"id": "a", "topic": "adk"}, {"id": "b", "topic": "workflow"}]
result, id_events, *_ = await run_workflow(stable, json.dumps(items), show_events=False)
paths = [event.node_info.path for event in id_events if event.node_info]
assert any("item-a" in value for value in paths)
print("执行路径：", *paths, sep="\n")


执行路径：
stable@1/stable_batch@1/process_item@item-a
stable@1/stable_batch@1/process_item@item-b
stable@1/stable_batch@1


### 11.3 Checkpoint 不是副作用的 exactly-once 保证

假设外部服务已经发出邮件，但进程在保存「已完成」事件之前崩溃。恢复时该节点可能再执行一次。**节点结果被记录，只能避免已知完成步骤的重复执行，不能消除这个窗口。**

生产中应把外部副作用封装为叶子节点，并使用业务系统支持的幂等键/事务，例如「业务订单 ID + 操作类型 + 版本」。`run_id` 是调度身份，不能自动替代外部服务的幂等键。

```python
# 示意：只有外部 API 本身支持幂等，这个键才有实际保障作用。
# await mail_service.send(payload, idempotency_key="article-42:publish:v1")
```

恢复还要求流程代码兼容：部署修改后，节点命名、调用顺序或输入语义发生变化时，不应盲目回放旧 invocation；应保留流程版本，或做明确迁移。


---

## 12. 综合项目：批量内容生产 + 有界并发 + 审核 + 人工确认

**场景**：输入一组主题，为每项生成一份技术介绍，规则不达标时最多修改一次；有任一项仍不达标则停止。全部达标后，把整批结果送人工确认，接受才模拟归档。

```mermaid
flowchart TD
    I["主题列表"] --> V["校验非空、数量上限、去重"]
    V --> P["parallel worker<br/>最多两个工作节点并发"]
    P --> W["每项：DeepSeek 写作"]
    W --> C{"本地检查"}
    C -->|未达标| R["DeepSeek 修改一次，再检查"]
    C -->|达标| G["汇总所有项"]
    R --> G
    G --> Q{"是否全部达标"}
    Q -->|否| F["返回 needs_revision，不进入发布"]
    Q -->|是| H["RequestInput 人工接受整批候选稿"]
    H -->|接受| A["模拟归档"]
    H -->|拒绝| N["返回 rejected"]
```

这里复用第 6 节的 `article_workflow`、第 10 节的 `ask_approval`。LLM 负责内容，Python 负责预算、准入和状态转移。设置最多 4 个不同主题：**正常业务流程每项至多 2 个 Agent 调用步骤，共最多 8 个步骤**；模型 SDK 或服务的网络重试另计，重跑新任务也会产生新调用。

代码单元格只定义流程，不自动产生 API 费用。随后两个示例调用分别演示离线规则版和需要 API Key 的 DeepSeek 版。


In [19]:
@node
def validate_topics(node_input) -> list[str]:
    topics = json.loads(input_text(node_input))
    if not isinstance(topics, list) or not 1 <= len(topics) <= 4:
        raise ValueError("请输入包含 1～4 个主题的 JSON 列表")
    if any(not isinstance(topic, str) or not topic.strip() for topic in topics):
        raise ValueError("主题必须是非空字符串")
    topics = [topic.strip() for topic in topics]
    if len(topics) != len(set(topics)):
        raise ValueError("主题不能重复")
    return topics


@node
def archive_batch(node_input: list[dict]) -> dict:
    # 演示只返回结果；没有创建文件，也没有发出外部发布请求。
    return {"status": "archived_demo", "count": len(node_input), "articles": node_input}


article_workers = node(
    article_workflow, parallel_worker=True, max_parallel_workers=2,
)


@node(rerun_on_resume=True)
async def batch_editor(ctx: Context, node_input):
    topics = await ctx.run_node(validate_topics, node_input)
    results = await ctx.run_node(article_workers, topics)
    if not all(item["passed"] for item in results):
        return {"status": "needs_revision", "articles": results}
    reply = await ctx.run_node(ask_approval, {"articles": results})
    if not ApprovalDecision.model_validate(reply).approved:
        return {"status": "rejected", "articles": results}
    return await ctx.run_node(archive_batch, results)


batch_app = Workflow(name="batch_app", edges=[("START", batch_editor)])
print("✅ DeepSeek 综合流程已定义")


✅ DeepSeek 综合流程已定义


### 12.1 先跑不依赖模型的版本，验证编排闭环

将内容节点换成确定性模拟节点，外层并发、准入与确认语义保持一致。这样可以在缺少密钥时理解完整流程，也便于把「编排错误」与「模型输出不达标」分开排查。


In [20]:
@node
def mock_article(node_input: str) -> dict:
    return {"draft": {"title": node_input, "body": "定义、示例、边界"},
            "passed": True, "findings": [], "revisions": 0}


mock_workers = node(mock_article, parallel_worker=True, max_parallel_workers=2)


@node(rerun_on_resume=True)
async def offline_batch_editor(ctx: Context, node_input):
    topics = await ctx.run_node(validate_topics, node_input)
    results = await ctx.run_node(mock_workers, topics)
    reply = await ctx.run_node(ask_approval, {"articles": results})
    if not ApprovalDecision.model_validate(reply).approved:
        return {"status": "rejected", "articles": results}
    return await ctx.run_node(archive_batch, results)


offline_batch = Workflow(name="offline_batch", edges=[("START", offline_batch_editor)])
_, offline_events, offline_runner, _ = await run_workflow(
    offline_batch, json.dumps(["节点", "checkpoint"]),
    session_id="offline_batch", show_events=False,
)
result, _ = await answer_approval(offline_runner, "offline_batch", offline_events, True)
assert result["status"] == "archived_demo" and result["count"] == 2
print("✅ 离线综合案例通过")


  📦 [offline_batch@1/offline_batch_editor@1/archive_batch@1] {'status': 'archived_demo', 'count': 2, 'articles': [{'draft': {'title': '节点', 'body': '定义、示例、边界'}, 'passed': True, 'findings': [], 'revisions': 0}, {'draft': {'title': 'checkpoint
  📦 [offline_batch@1/offline_batch_editor@1] {'status': 'archived_demo', 'count': 2, 'articles': [{'draft': {'title': '节点', 'body': '定义、示例、边界'}, 'passed': True, 'findings': [], 'revisions': 0}, {'draft': {'title': 'checkpoint
✅ 离线综合案例通过


### 12.2 再运行 DeepSeek 版

本单元格会调用模型。若返回 `needs_revision`，说明字数等确定性规则仍不满足，应查看 `findings`，不进入确认；若结果为 `None` 且有 `RequestInput`，说明正常等待人工确认。

为避免教学代码代替用户接受整批内容，下面只运行到暂停点并展示候选稿。接受/拒绝后的恢复方式沿用第 10 节；把真实用户决定传给 `answer_approval(...)` 即可。


In [21]:
if os.getenv("DEEPSEEK_API_KEY"):
    batch_result, batch_events, batch_runner, batch_ss = await run_workflow(
        batch_app, json.dumps(["ADK 动态工作流", "工作流 checkpoint"]),
        session_id="deepseek_batch",
    )
    if batch_result is None:
        invocation_id, call = pending_input(batch_events)
        print("⏸️ 候选稿等待人工确认；invocation_id =", invocation_id)
        print("请求内容：", json.dumps(call.args, ensure_ascii=False, indent=2))
    else:
        print("🏁", json.dumps(batch_result, ensure_ascii=False, indent=2))
else:
    print("⏭️ 跳过 DeepSeek 综合调用；上面的离线流程可完整运行。")


  📦 [batch_app@1/batch_editor@1/validate_topics@1] ['ADK 动态工作流', '工作流 checkpoint']
  🗣️ [batch_app@1/batch_editor@1/article_workflow@1/article_workflow@2/dynamic_writer@1] {"title": "工作流 checkpoint：让长任务可恢复", "body": "工作流 checkpoint 是在流程执行中保存的中间状态快照，便于失败后从断点续跑，而不是从头再来。例如批量处理 1 万条订单时，每完成 1000 
  📦 [batch_app@1/batch_editor@1/article_workflow@1/article_workflow@2/check_article@1] {'passed': True, 'findings': []}
  📦 [batch_app@1/batch_editor@1/article_workflow@1/article_workflow@2] {'draft': {'title': '工作流 checkpoint：让长任务可恢复', 'body': '工作流 checkpoint 是在流程执行中保存的中间状态快照，便于失败后从断点续跑，而不是从头再来。例如批量处理 1 万条订单时，每完成 1000 条就记录一次进度；若第 7000 条报错，重启后直接从第 7000 条继续，省时又省资源。'}, '
  🗣️ [batch_app@1/batch_editor@1/article_workflow@1/article_workflow@1/dynamic_writer@1] {"title": "ADK 动态工作流入门", "body": "ADK 动态工作流是指由智能体根据运行时上下文自主决定下一步执行哪个子智能体或工具，而非预先写死流程。例如，客服入口智能体收到“查订单”请求，会动态转交订单查询智能体；若用
  📦 [batch_app@1/batch_editor@1/article_workflow@1/article_workflow@1/check_article@1] {'passed': False, 'findings': ['正文有 13

---

## 13. 与 LangGraph 对照 🔄

第 4 章主要把 ADK 模板 Agent 映射到 LangGraph 图；现在有了代码式动态编排，更适合同时对照 LangGraph 的 **Graph API 与 Functional API**。

| 本章 ADK 构件 | LangGraph 对照 | 类比的边界 |
|---|---|---|
| `Workflow(edges=...)` | `StateGraph` 的 nodes / edges | 状态模型与路由协议不相同 |
| 动态编排节点 + `ctx.run_node` | Functional API 的 entrypoint / task | 都可在代码中表达流程；装饰器与恢复协议不同 |
| `node_input` / `Event.output` | 节点/任务输入输出 | 不等于所有值都自动写进全局 State |
| `ctx.state` | State / 持久化数据 | 并发更新需要各自设计；ADK 没有自动套用 LangGraph reducer |
| `RequestInput` + FunctionResponse | `interrupt()` + `Command(resume=...)` | 恢复输入格式不同，不能直接互换 |
| Session 事件中的 checkpoint | Checkpointer 保存的执行状态 | 都要考虑存储与重放；持久化配置不同 |
| `run_id` 与调用顺序 | 恢复时任务匹配与重放顺序 | 是执行身份，不是业务 exactly-once 保证 |

**选型思路**：业务步骤与路径适合画图，优先显式图；大量数据驱动循环、局部条件与函数复用，考虑动态编排；开放式专家选择才需要模型委派。需要部署可恢复的长期任务时，还应单独评估存储、流程版本、并发限额与故障语义。

参考：[LangGraph Functional API](https://docs.langchain.com/oss/python/langgraph/functional-api) · [Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution)。

---

## 14. 常见错误速查

| 现象 | 可能原因 | 修正方向 |
|---|---|---|
| DeepSeek 返回 `This response_format type is unavailable now` | 将 `output_schema` 转成了不支持的 `json_schema` | 使用 `json_object` + 本地 Pydantic 校验 |
| 动态节点调用时报必须 rerun 的错误 | 编排节点缺少 `rerun_on_resume=True` | 每一层调度父节点都设置 |
| 提示从 State 找不到参数 | 把 `node_input` 改成了普通参数名 | 保持接口名或用显式字段绑定 |
| 首节点 `.strip()` 报错 | 把 `Content` 当字符串 | 在 Runner 边界明确解包 |
| 只得到中间节点结果 | 只靠 `is_final_response()` 筛结果 | 看根节点的 `output_for` |
| 并行分支结果被覆盖 | 都写相同 State key | 返回结果，由父节点汇总 |
| 暂停被当成失败或拒绝 | 捕获暂停信号，或忽略 pending input | 匹配真实请求并恢复同一 invocation |
| 恢复后再次发邮件/扣款 | 副作用未封装或无外部幂等 | 叶子节点 + 外部业务幂等 |
| 回放拿到错误的旧结果 | ID、输入或调用顺序不一致 | 固定任务快照，保留流程版本 |
| 超时/重试后假装成功 | 吞异常并返回普通结果 | 限定可重试错误，显式降级状态 |

---

## 📌 本章要点回顾

- **Dynamic workflows 是程序化编排**：动态路径由 Python 表达，是否让 LLM 作决策是另一个问题；
- `Workflow` 对接 Runner，`@node` 注册节点，`ctx.run_node()` 调度并等待子节点；
- 用返回值传递局部数据，用 schema 校验契约，需要共享时才写 `ctx.state`；
- 条件、循环、并发和嵌套都要有明确的预算与失败语义；
- 编排父节点重入、完成子节点结果回放，共同构成恢复机制；
- checkpoint、execution ID 与外部业务幂等各有职责，不能互相替代。

> 🔄 回看：[04-多智能体与工作流](04-多智能体与工作流.ipynb)，比较预设流程与本章的代码编排；再对照 [LangGraph 核心：图与状态](../langchain-langgraph/04-LangGraph核心-图与状态.ipynb)。


---

## 🧪 本章练习

### 1. 用 Dynamic workflow 重写内容流水线（基础）

把第 4 章的「大纲 → 初稿 → 润色」改为动态节点调用，使用明确的输入输出传值。打印 `node_info.path`，证明三步顺序执行；指出哪些 State key 已不再必要，哪些摘要仍值得保留。

### 2. 可验收的迭代改进（进阶）

把第 5 节规则检查改为「标题非空、正文长度范围、必须包含指定主题词」。独立设置最多修改次数，覆盖一次通过、修订后通过、达到上限仍失败三种结果；失败时不得把最后稿标为合格。再换为 DeepSeek 修订，比较模型不稳定输出与编排逻辑是否独立。

### 3. 有上限的动态批处理（进阶）

输入 1～20 项，使用 parallel worker 设置最大并发 3。记录实际同时运行节点数，证明并发从未超过上限；加入一个故障项，明确选择整批失败或部分成功，并设计不会吞掉人工暂停信号的处理方式。

### 4. 暂停与恢复的证据（工程）

实现「读取计划 → 三路分析 → 人工确认 → 模拟归档」。用函数执行计数证明确认后只执行未完成步骤；覆盖接受与拒绝。进一步将会话事件持久化，重新构造运行时后恢复同一 invocation，说明流程定义、Session 存储与调用身份分别需要保留什么。

### 5. Execution ID 与副作用边界（工程综合）

输入一批具有唯一业务 ID 的对象，设计稳定排序或显式 run_id；拒绝重复 ID。模拟外部写入成功但 checkpoint 尚未记录的故障窗口，用业务幂等键证明不会重复写入。再说明为什么仅设置自定义 run_id 无法解决这个窗口。

### 6. 两种表达方式的取舍（跨框架）

选综合项目画一张 ADK graph 或 LangGraph StateGraph，再写出 Functional API 式控制流草图。比较可读性、恢复顺序、状态耦合与可观测性。必须区分「流程写在代码里」与「模型自主规划」，不能只按 API 名称作判断。

---

## 📚 官方资料与版本核对

- [Dynamic agent workflows](https://adk.dev/graphs/dynamic/)：概念、动态路由与 execution ID；
- [Graph-based workflows](https://adk.dev/graphs/) / [Graph routes](https://adk.dev/graphs/routes/)：图式控制流与组合；
- [Data handling](https://adk.dev/graphs/data-handling/)：节点输出、State 与 schema；
- [Human input](https://adk.dev/graphs/human-input/)：人工输入节点；
- [Custom workflows](https://adk.dev/agents/custom-agents/)：与旧 `BaseAgent` 自定义编排的关系；
- [Python API Reference](https://adk.dev/api-reference/python/)：在线参考可能比本仓库更新；
- [ADK Python v2.6.3 源码](https://github.com/google/adk-python/tree/v2.6.3/src/google/adk)：本章的接口与实现核对基准。

本章已在仓库的 2.6.3 环境运行全部无模型示例。DeepSeek 兼容性修复后，在真实 ADK→LiteLLM 请求参数转换层验证了 `json_object`、写作/修改与批量确认恢复，并使用真实 DeepSeek 服务运行一次文章流水线，JSON 解析、字段校验和业务检查均通过。模型输出仍有不确定性，后续运行是否达标应以实际检查结果为准。
